# Notebook 3 — RNN: Feature Extraction, Preprocessing & Training

## 1 Setup

In [3]:
import sys
print(sys.executable)
print(sys.path)
%pip install pandas matplotlib numpy tensorflow nltk ipykernel

/home/lukas/ai_project/venv/bin/python
['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/home/lukas/ai_project/venv/lib/python3.12/site-packages']
ERROR: Operation cancelled by user
^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import json, random, sys, time

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.wajib.rnn.RNN import buildRNNKeras, trainRNNKeras, trainRNNDataset, RNNScratch
from src.wajib.shared.layers import EmbeddingLayer, DenseLayer
from src.wajib.shared.preprocessing import (
    loadFlickr8kCaptions, buildVocabulary, saveVocabulary, loadVocabulary,
)
from src.wajib.shared.decoder import greedyDecode
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("GPUs:", gpus)

2026-05-14 07:25:05.823501: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-14 07:25:06.020916: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778718306.334542   14680 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778718306.426646   14680 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778718306.561995   14680 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2 Config

In [ ]:
FEATURES_NPY  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_features.npy"
FEATURES_IDX  = PROJECT_ROOT / "src/wajib/weights/features/flickr8k_index.json"
CAPTIONS_FILE = PROJECT_ROOT / "data/flickr8k/captions.txt"
VOCAB_PATH    = PROJECT_ROOT / "src/wajib/weights/vocab.json"
WEIGHTS_DIR   = PROJECT_ROOT / "src/wajib/weights/rnn"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

EMBED_DIM    = 256
MAX_LEN      = 30
EPOCHS       = 5
BATCH_SIZE   = 64
CNN_FEAT_DIM = 2048

# 3 variasi num_layers × 2 variasi hidden_dim = 6 total
VARIATIONS = [
    (1, 128),
    (1, 512),
    (2, 128),
    (2, 512),
    (3, 128),
    (3, 512),
]

### 3 Load CNN Features

In [ ]:
features_matrix = np.load(FEATURES_NPY)
with open(FEATURES_IDX) as f:
    idx_names = json.load(f)

image_features = {name: features_matrix[i] for i, name in enumerate(idx_names)}
print(f"Loaded {len(image_features)} features, dim={features_matrix.shape[1]}")

### 4 Load Captions + Split + Vocab

In [ ]:
captions_dict = loadFlickr8kCaptions(str(CAPTIONS_FILE))

all_images = list(captions_dict.keys())
random.shuffle(all_images)  # seed already set in cell 1

train_imgs = set(all_images[:6000])
val_imgs   = set(all_images[6000:7000])
test_imgs  = set(all_images[7000:])

train_caps = {k: v for k, v in captions_dict.items() if k in train_imgs}
val_caps   = {k: v for k, v in captions_dict.items() if k in val_imgs}
test_caps  = {k: v for k, v in captions_dict.items() if k in test_imgs}

print(f"Split => train={len(train_caps)}, val={len(val_caps)}, test={len(test_caps)}")

if VOCAB_PATH.exists():
    vocab = loadVocabulary(str(VOCAB_PATH))
    print(f"Vocab loaded: {len(vocab)}")
else:
    all_train_caps = [cap for caps in train_caps.values() for cap in caps]
    vocab = buildVocabulary(all_train_caps, min_freq=2)
    saveVocabulary(vocab, str(VOCAB_PATH))
    print(f"Vocab built + saved: {len(vocab)}")

### 5 Build Dataset Arrays

In [ ]:
X_cnn_tr, X_tok_tr, y_tr = trainRNNDataset(image_features, train_caps, vocab, MAX_LEN)
X_cnn_va, X_tok_va, y_va = trainRNNDataset(image_features, val_caps,   vocab, MAX_LEN)

print(f"Train -> X_cnn={X_cnn_tr.shape}, X_tok={X_tok_tr.shape}, y={y_tr.shape}")
print(f"Val   -> X_cnn={X_cnn_va.shape}, X_tok={X_tok_va.shape}, y={y_va.shape}")

### 6 Training (6 Variation)

In [ ]:
histories = {}

for num_layers, hidden_dim in VARIATIONS:
    name      = f"rnn_{num_layers}L_{hidden_dim}h"
    save_path = WEIGHTS_DIR / f"{name}.keras"

    print(f"\n{'='*50}\n  {name}\n{'='*50}")

    model = buildRNNKeras(
        vocab_size     = len(vocab),
        embed_dim      = EMBED_DIM,
        hidden_dim     = hidden_dim,
        num_rnn_layers = num_layers,
        cnn_feature_dim= CNN_FEAT_DIM,
    )
    model.summary()

    hist = trainRNNKeras(
        model,
        X_cnn_tr, X_tok_tr, y_tr,
        X_cnn_va, X_tok_va, y_va,
        epochs     = EPOCHS,
        batch_size = BATCH_SIZE,
        save_path  = str(save_path),
    )
    histories[name] = hist
    print(f"  Saved to {save_path.name}")

print(f"\nSemua {len(VARIATIONS)} variasi selesai.")

### Training and Loss Curve

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (name, hist) in zip(axes.flatten(), histories.items()):
    ax.plot(hist['loss'],     label='train')
    ax.plot(hist['val_loss'], label='val')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

plt.suptitle("RNN Training Loss per Variasi", y=1.01)
plt.tight_layout()
plt.savefig(WEIGHTS_DIR / "rnn_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

### Training Summary

In [ ]:
print(f"{'Model':<22} {'Best Val Loss':>14} {'Epochs':>7}")
print('-' * 45)
for name, hist in histories.items():
    print(f"{name:<22} {min(hist['val_loss']):>14.4f} {len(hist['val_loss']):>7}")

best_name   = min(histories, key=lambda n: min(histories[n]['val_loss']))
best_layers = int(best_name.split('L')[0].split('_')[1])
print(f"\nBest: {best_name}")

In [ ]:

def loadScratchFromKeras(model_path, num_layers):
    model = tf.keras.models.load_model(str(model_path))

    embed = EmbeddingLayer()
    embed.loadWeights(model.get_layer('embedding'))

    proj = DenseLayer()
    proj.loadWeights(model.get_layer('cnn_proj'))

    out = DenseLayer(activation='softmax')
    out.loadWeights(model.get_layer('output'))

    rnn = RNNScratch()
    rnn.loadWeights([model.get_layer(f'rnn_{i}') for i in range(num_layers)])

    return rnn, proj, embed, out

In [ ]:
# Decode Testing (3 sampel)
best_path = WEIGHTS_DIR / f"{best_name}.keras"
rnn, proj, embed, out = loadScratchFromKeras(best_path, best_layers)

for img in list(val_caps.keys())[:3]:
    feat    = image_features[img]
    caption = ' '.join(greedyDecode(rnn, proj, embed, out, feat, vocab))
    gt      = val_caps[img][0]
    print(f"GT  : {gt}")
    print(f"Pred: {caption}")
    print()

## Analisis (BLEU-4)

In [ ]:
results = {}

for num_layers, hidden_dim in VARIATIONS:
    name = f"rnn_{num_layers}L_{hidden_dim}h"
    path = WEIGHTS_DIR / f"{name}.keras"
    rnn, proj, embed, out = loadScratchFromKeras(path, num_layers)

    bleu_scores = []
    t0 = time.time()

    for img, caps in test_caps.items():
        if img not in image_features:
            continue
        hyp  = greedyDecode(rnn, proj, embed, out, image_features[img], vocab, MAX_LEN)
        refs = [cap.split() for cap in caps]
        bleu_scores.append(sentence_bleu(refs, hyp, weights=(0.25, 0.25, 0.25, 0.25)))

    elapsed = time.time() - t0
    results[name] = {'bleu4': np.mean(bleu_scores), 'time_s': elapsed,
                    'layers': num_layers, 'hidden': hidden_dim}
    print(f"{name:<22}  BLEU-4={results[name]['bleu4']:.4f}  time={elapsed:.1f}s")

### Keras Greedy Decode for Best Model

In [ ]:
def kerasGreedyDecode(model, cnn_feat, vocab, max_len=MAX_LEN):
    id2word = {v: k for k, v in vocab.items()}
    tokens  = [vocab['<start>']]
    for _ in range(max_len):
        out_k = model.predict(
            [cnn_feat[np.newaxis], np.array([tokens])], verbose=0
        )
        nxt = int(np.argmax(out_k[0, -1]))
        if nxt == vocab['<end>']:
            break
        tokens.append(nxt)
    return [id2word.get(t, '<unk>') for t in tokens[1:]]

best_keras_model = tf.keras.models.load_model(str(WEIGHTS_DIR / f"{best_name}.keras"))
rnn, proj, embed, out = loadScratchFromKeras(WEIGHTS_DIR / f"{best_name}.keras", best_layers)

bleu_keras, bleu_scratch = [], []
t_keras = t_scratch = 0.0

for img, caps in test_caps.items():
    if img not in image_features:
        continue
    feat = image_features[img]
    refs = [cap.split() for cap in caps]

    t0 = time.time()
    hyp_k = kerasGreedyDecode(best_keras_model, feat, vocab)
    t_keras += time.time() - t0
    bleu_keras.append(sentence_bleu(refs, hyp_k, weights=(0.25, 0.25, 0.25, 0.25)))

    t0 = time.time()
    hyp_s = greedyDecode(rnn, proj, embed, out, feat, vocab, MAX_LEN)
    t_scratch += time.time() - t0
    bleu_scratch.append(sentence_bleu(refs, hyp_s, weights=(0.25, 0.25, 0.25, 0.25)))

print(f"Keras   BLEU-4={np.mean(bleu_keras):.4f}  total={t_keras:.1f}s")
print(f"Scratch BLEU-4={np.mean(bleu_scratch):.4f}  total={t_scratch:.1f}s")

### Max Caption Length Variation

In [ ]:
rnn, proj, embed, out = loadScratchFromKeras(WEIGHTS_DIR / f"{best_name}.keras", best_layers)

print(f"{'max_len':>8} {'BLEU-4':>10}")
print('-' * 20)

for ml in [15, 30, 50]:
    scores = [
        sentence_bleu(
            [cap.split() for cap in caps],
            greedyDecode(rnn, proj, embed, out, image_features[img], vocab, ml),
            weights=(0.25, 0.25, 0.25, 0.25)
        )
        for img, caps in test_caps.items()
        if img in image_features
    ]
    print(f"{ml:>8} {np.mean(scores):>10.4f}")

# Final Results

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Model": name,
        "Layers": r["layers"],
        "Hidden": r["hidden"],
        "BLEU-4": round(r["bleu4"], 4),
        "Time(s)": round(r["time_s"], 1),
    })

df = pd.DataFrame(rows)

df = df.sort_values("BLEU-4", ascending=False)

print(df.to_string(index=False))